In [0]:
import requests
import pandas as pd
import json
from pyspark.sql import SparkSession
import datetime
import hashlib

In [0]:
def get_data_by_api(url):
    data = []

    page_num = 1

    while True:
        response = requests.get(url, params={"page": page_num, "per_page": 100})
        
        if response.status_code == 200:
            page = response.json()
            if not page:
                break
            data.extend(page)
            page_num += 1
        else:
            print(f"Erro na requisição: {response.status_code}")
            break
    return data

In [0]:
def gerar_hash(data):
    """Gera um hash SHA-256 para o conteúdo de um JSON já carregado."""
    try:
        json_serial = json.dumps(data, sort_keys=True, ensure_ascii=False)
        hash_sha256 = hashlib.sha256(json_serial.encode('utf-8')).hexdigest() 
        return hash_sha256
    except Exception as e:
        print(f"Erro ao gerar hash: {e}")
        return None


In [0]:
def check_hash(volume_path, hash):
    try:
        spark = SparkSession.builder.getOrCreate()
        arquivos = [f.name for f in dbutils.fs.ls(volume_path)]
        
        # hash = gerar_hash(json_data)
        if hash is None:
            return False
        return hash in arquivos
    
    except Exception as e:
        print(f"Erro ao verificar hash no volume: {e}")
        return False

In [0]:
# now = datetime.datetime.now().strftime("%Y%m%d_%H%M")
url = "https://api.openbrewerydb.org/v1/breweries"

data = get_data_by_api(url)
hash = gerar_hash(data)

if(check_hash("/Volumes/brewery/bronze_db/raw_data/breweries", data)):
    print(check_hash("/Volumes/brewery/bronze_db/raw_data/breweries", data))
    path = f"/Volumes/brewery/bronze_db/raw_data/breweries/{gerar_hash(data)}.json"
    with open(path, "w") as f:
        json.dump(data, f)

In [0]:
# dbutils.fs.mkdirs("/Volumes/brewery/bronze_db/raw_data/breweries")
